# Lesson 05 — Drawing Object Bounding Box in a Scene

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Query: the object you want to find
# Scene: the image where you want to find it
query = cv2.imread('sample.jpg')
scene = query.copy()  # in reality: different image containing the object

# Simulate: scene is the query placed in a different context
h_q, w_q = query.shape[:2]
M = cv2.getRotationMatrix2D((w_q//2,h_q//2),15,0.7)
scene = cv2.warpAffine(scene, M, (scene.shape[1]+100, scene.shape[0]+100))

sift = cv2.SIFT_create()
kp_q,d_q = sift.detectAndCompute(cv2.cvtColor(query,cv2.COLOR_BGR2GRAY),None)
kp_s,d_s = sift.detectAndCompute(cv2.cvtColor(scene,cv2.COLOR_BGR2GRAY),None)
good = [m for m,n in cv2.BFMatcher().knnMatch(d_q,d_s,k=2) if m.distance<0.75*n.distance]

if len(good) >= 4:
    src = np.float32([kp_q[m.queryIdx].pt for m in good]).reshape(-1,1,2)
    dst = np.float32([kp_s[m.trainIdx].pt for m in good]).reshape(-1,1,2)
    H, mask = cv2.findHomography(src, dst, cv2.RANSAC, 5.0)

    # Project the 4 corners of the query into the scene
    corners = np.float32([[0,0],[w_q-1,0],[w_q-1,h_q-1],[0,h_q-1]]).reshape(-1,1,2)
    scene_corners = cv2.perspectiveTransform(corners, H)

    vis = scene.copy()
    cv2.polylines(vis, [np.int32(scene_corners)], True, (0,255,0), 4)
    cv2.putText(vis, 'FOUND', tuple(np.int32(scene_corners[0][0])+np.array([5,20])),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)

    plt.figure(figsize=(14,6))
    plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    plt.title(f'Object found in scene — {mask.sum()} inlier matches'); plt.axis('off'); plt.show()

## Key Takeaway
`perspectiveTransform(corners, H)` projects points through the homography.
This maps the object's boundary from query image into scene coordinates.
This is exactly how AR marker detection works.